# Fatiga cognitiva: análisis estadístico longitudinal
**Versión 3 · análisis principal estadístico; predicción exploratoria en `fatigue_analysis_clean_v2.ipynb`.**

**Pregunta principal:** ¿se asocia Glu/tCr PFC de la sesión actual con la fatiga de la siguiente sesión, una vez ajustadas la fatiga actual, la sesión y la condición?

El análisis prioriza tamaño de asociación e incertidumbre. No intenta mejorar R² ni interpreta una asociación como causalidad o capacidad de anticipar fatiga antes de cualquier reporte consciente. Es exploratorio: la hipótesis y el procedimiento se reformularon después de inspeccionar análisis anteriores; no es un estudio confirmatorio preregistrado.

1. Configuración, seis fuentes y trazabilidad.
2. Features por sesión y transiciones estrictamente adyacentes.
3. Descripción, disponibilidad y visualización de las modalidades.
4. Auditoría de outliers para sensibilidad.
5. Modelo longitudinal principal e intervalos de confianza.
6. t de Student pareada: contraste complementario del cambio de fatiga.
7. Sensibilidad a exclusiones, remuestreo por sujeto y separación intra/interindividual.
8. Análisis complementario de las demás modalidades con control de multiplicidad.
9. Exportaciones e interpretación.

**Decisión sobre valores extremos:** la inferencia principal conserva todas las transiciones temporalmente válidas con las variables del modelo observadas. Eliminar registros por su cambio de fatiga condiciona el análisis al resultado y puede sesgar asociaciones. Las exclusiones solicitadas se ejecutan y documentan en la sensibilidad, incluyendo todos los candidatos por encima del umbral. La comparación muestra cuánto dependen las conclusiones de esa limpieza.

Las tres fuentes adicionales se procesan cuando están disponibles; su ausencia se declara. El CSV de pupila puede suministrarse como `.csv.gz`. Ninguna modalidad ausente se reemplaza con datos ficticios ni bloquea el modelo principal, que utiliza ratings y semiLASER. N-task también se carga para conservar el análisis conductual y de calidad.

## 1. Configuración y archivos
Ejecuta desde un kernel con `numpy pandas scipy matplotlib seaborn statsmodels scikit-learn`. Para reproducir los resultados anteriores se usaron Python 3.12.14, NumPy 2.5.4, pandas 3.1.0, SciPy 1.18.1, statsmodels 0.15.0 y scikit-learn 1.9.1; el manifiesto guarda las versiones realmente utilizadas.

La carpeta `data/` tiene prioridad sobre `/data`. Se aceptan CSV, gzip y ZIP con un solo CSV. Las fuentes nucleares son ratings, n-task y semiLASER; DWS, pupila y choice-bias se incorporan si están presentes. Todos los archivos originales se conservan. Los parámetros de choice-bias requieren verificar que su estimación original no utilizó resultados futuros.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Prefer the repository's data/ folder shown in the project, then /data.
DATA_DIR = Path("data") if Path("data").is_dir() else Path("/data")
# Includes externally fitted parameters in an explicitly exploratory all-six-files model.
# Set False if their estimation used other sessions, future outcomes or held-out participants.
INCLUDE_CHOICE_PARAMETERS = True
RESULTS_DIR = Path("fatigue_longitudinal_results")
SEED = 42
N_SPLITS = 5
# Conservative robust-z rules; sensitivity cases below vary the burden cutoff.
OUTLIER_Z = 3.5
SUBJECT_BURDEN = 0.20
MIN_OBSERVED_CELLS = 8
EXTREME_TRANSITION_Z = 4.5
MAX_MISSING = 0.40
MAX_CORRELATION = 0.90
sns.set_theme(style="whitegrid", context="notebook")
FILES = {
    "ratings": "MRSFATIGUE_ratings_session_zenodo.csv",
    "trials": "MRSFATIGUE_ntask_session_zenodo.csv",
    "mrs": "MRSFATIGUE_MRS_semiLaser_session_ratio_Glu_tCr_zenodo.csv",
    "dws": "MRSFATIGUE_MRS_DWS_session_ADC_log_Glu_Gln_lin_zenodo.csv",
    "pupil": "MRSFATIGUE_pupil_choice_session_zenodo.csv",
    "choice": "MRSFATIGUE_choice_bias_parameter_zenodo.csv",
}
VERSIONS = {p: metadata.version(p) for p in
            ["numpy", "pandas", "scipy", "matplotlib", "seaborn", "scikit-learn"]}
print("Python", platform.python_version(), VERSIONS)
import warnings
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
VERSIONS["statsmodels"] = metadata.version("statsmodels")
BOOTSTRAP_REPEATS = 2000
PAIR_START = 0
PAIR_END = 5
MIN_SCREEN_SUBJECTS = 15
MIN_SCREEN_ROWS = 20

In [ ]:
# Read plain CSV or losslessly compressed CSV; pandas infers gzip/ZIP compression.
INPUT_PATHS = {}
for name, filename in FILES.items():
    candidates = [DATA_DIR / filename, DATA_DIR / (filename + ".gz"),
                  DATA_DIR / (filename + ".zip"), (DATA_DIR / filename).with_suffix(".zip")]
    selected = next((path for path in candidates if path.is_file()), None)
    if selected is not None:
        INPUT_PATHS[name] = selected
CORE_SOURCES = {"ratings", "trials", "mrs"}
missing_files = [filename for name, filename in FILES.items()
                 if name in CORE_SOURCES and name not in INPUT_PATHS]
missing_optional = [name for name in FILES if name not in INPUT_PATHS and name not in CORE_SOURCES]
print("Fuentes complementarias ausentes:", missing_optional or "ninguna")
if missing_files:
    raise FileNotFoundError("Raw CSV or compressed CSV files are required. Set DATA_DIR. Missing: " + ", ".join(missing_files))
raw = {name: pd.read_csv(path) for name, path in INPUT_PATHS.items()}
provenance = {name: {"file": str(INPUT_PATHS[name]),
                    "sha256": hashlib.sha256(INPUT_PATHS[name].read_bytes()).hexdigest(),
                    "rows": len(raw[name])} for name in INPUT_PATHS}
KEY = ["subject_id", "condition", "session"]
REQUIRED = {
    "ratings": KEY + ["question", "rating"],
    "trials": KEY + ["block", "trial", "performance", "response_time", "rule_switch"],
    "mrs": KEY + ["ROI", "Glu_tCr"],
    "dws": KEY + ["ROI", "ADC_log_Glu_Gln_lin", "WCONC_corr", "p_shots_kept", "SN", "linewidth"],
    "pupil": KEY + ["domain", "response_time", "choiceSS", "ntask_perf"],
    "choice": KEY + ["domain", "model", "parameter_name", "parameter_value"],
}
for name, columns in REQUIRED.items():
    if name not in raw:
        continue
    absent = set(columns) - set(raw[name].columns)
    if absent:
        raise ValueError(f"{name}: missing columns {sorted(absent)}")
    if raw[name][KEY].isna().any().any():
        raise ValueError(f"{name}: missing join keys")
    raw[name]["subject_id"] = raw[name]["subject_id"].astype(str)
    raw[name]["condition"] = raw[name]["condition"].astype(str).str.strip()
    raw[name]["session"] = pd.to_numeric(raw[name]["session"], errors="raise")
    if not (raw[name]["session"] % 1 == 0).all():
        raise ValueError("Session must be an integer ordinal, not a label with unknown order.")
    raw[name]["session"] = raw[name]["session"].astype(int)
for name, columns in {"ratings": ["rating"], "trials": ["performance", "response_time", "rule_switch"],
                      "mrs": ["Glu_tCr"],
                      "dws": ["ADC_log_Glu_Gln_lin", "WCONC_corr", "p_shots_kept", "SN", "linewidth"],
                      "choice": ["parameter_value"]}.items():
    if name not in raw:
        continue
    for column in columns:
        raw[name][column] = pd.to_numeric(raw[name][column], errors="raise").replace([np.inf, -np.inf], np.nan)
display(pd.DataFrame(provenance).T)

## 2. Construcción temporal y control de claves
Una fila representa sujeto × condición × sesión. Las respuestas de Fatigue duplicadas en una misma clave se excluyen por ambigüedad temporal y se guardan aparte. No se promedian ni se unen condiciones diferentes. Solo se conserva t→t+1; nunca se saltan sesiones faltantes.

N-task se ordena por bloque y trial. Se calculan precisión, RT, variabilidad, switching y cambios entre mitades. Los RT no positivos pasan a ausentes y se cuentan. La agrupación de trials no los convierte en sujetos independientes. Las variables de sesión se interpretan como disponibles al terminar esa sesión, sujeto a verificar el protocolo de adquisición.

In [ ]:
trials = raw["trials"].copy()
for column in ["performance", "rule_switch"]:
    if not trials[column].dropna().isin([0, 1]).all():
        raise ValueError(f"{column} must be binary 0/1; verify source coding.")
if trials.duplicated(KEY + ["block", "trial"]).any():
    raise ValueError("Repeated trial keys: resolve task/trial identity before aggregation.")
invalid_rt = trials["response_time"].le(0)
print("Nonpositive reaction times set to missing:", int(invalid_rt.sum()))
trials.loc[invalid_rt, "response_time"] = np.nan
trials = trials.sort_values(KEY + ["block", "trial"])
group = trials.groupby(KEY, sort=False)
position = group.cumcount()
trials["half"] = np.where(position < group["trial"].transform("size") / 2, "early", "late")
behavior = group.agg(accuracy=("performance", "mean"), median_rt=("response_time", "median"),
                     mean_rt=("response_time", "mean"), rt_std=("response_time", "std"),
                     switch_fraction=("rule_switch", "mean"), n_trials=("trial", "size")).reset_index()
for half in ["early", "late"]:
    part = trials.loc[trials["half"].eq(half)].groupby(KEY).agg(
        **{f"{half}_accuracy": ("performance", "mean"), f"{half}_rt": ("response_time", "mean")}
    ).reset_index()
    behavior = behavior.merge(part, on=KEY, validate="one_to_one")
behavior["within_session_rt_change"] = behavior["late_rt"] - behavior["early_rt"]
behavior["within_session_accuracy_change"] = behavior["late_accuracy"] - behavior["early_accuracy"]

fatigue = raw["ratings"].loc[raw["ratings"]["question"].eq("Fatigue"), KEY + ["rating"]].copy()
ambiguous = fatigue.loc[fatigue.duplicated(KEY, keep=False)].copy()
fatigue = fatigue.loc[~fatigue.duplicated(KEY, keep=False)].rename(columns={"rating": "fatigue"})
# Both endpoints must have unambiguous ratings; a missing intermediate session cannot create a jump.
fatigue = fatigue.sort_values(KEY)
for source, dest in [("fatigue", "fatigue_future"), ("session", "session_future")]:
    fatigue[dest] = fatigue.groupby(["subject_id", "condition"])[source].shift(-1)
transitions = fatigue.loc[(fatigue["session_future"] - fatigue["session"]).eq(1)].dropna(
    subset=["fatigue", "fatigue_future"]).copy()
transitions["delta_fatigue"] = transitions["fatigue_future"] - transitions["fatigue"]

mrs = raw["mrs"].copy()
if mrs.duplicated(KEY + ["ROI"]).any():
    raise ValueError("Repeated MRS ROI keys: do not average unknown repeats.")
mrs_wide = mrs.pivot(index=KEY, columns="ROI", values="Glu_tCr")
mrs_wide.columns = [f"MRS_Glu_tCr_{roi}" for roi in mrs_wide.columns]
mrs_wide = mrs_wide.reset_index()
if "MRS_Glu_tCr_PFC" not in mrs_wide:
    raise ValueError("PFC glutamate is required for M2 and M3.")
sessions = fatigue[KEY + ["fatigue"]].merge(behavior, on=KEY, how="left", validate="one_to_one")
sessions = sessions.merge(mrs_wide, on=KEY, how="left", validate="one_to_one")
model_data = transitions.merge(behavior, on=KEY, how="left", validate="one_to_one").merge(
    mrs_wide, on=KEY, how="left", validate="one_to_one").reset_index(drop=True)
model_data["row_id"] = np.arange(len(model_data))
assert not model_data.duplicated(KEY).any()
assert (model_data["session_future"] == model_data["session"] + 1).all()
print("Ambiguous rating rows excluded:", len(ambiguous))
print("Transitions:", len(model_data), "; subjects:", model_data["subject_id"].nunique())
display(model_data.groupby(["session", "session_future"]).size().rename("n").reset_index())

### Integración de las seis fuentes
Se conservan las identidades de ROI, dominio, modelo y parámetro al pivotar. MRS y DWS incluyen sus métricas de calidad; pupila resume todos los samples numerados y sus cambios por orden de muestra, sin inventar frecuencia de muestreo ni alineación a eventos. ChoiceSS exige codificación binaria. La disponibilidad por modalidad y el inventario de columnas quedan exportados.

Estas features adicionales no se añaden juntas al modelo principal. Su análisis complementario incorpora una variable a la vez con ajuste por covariables y corrección Holm; se consideran hipótesis exploratorias.

In [ ]:
def numeric_values(frame, columns):
    """Convert documented measurements strictly; invalid strings need a data-dictionary decision."""
    result = frame.copy()
    for column in columns:
        result[column] = pd.to_numeric(result[column], errors="raise").replace([np.inf, -np.inf], np.nan)
    return result


def wide_features(frame, dimensions, values, prefix):
    """Keep ROI/domain/model/parameter identities distinct; never average unknown repeats."""
    full_key = KEY + dimensions
    if frame[full_key].isna().any().any():
        raise ValueError(f"{prefix}: missing feature identity")
    if frame.duplicated(full_key).any():
        raise ValueError(f"{prefix}: duplicate keys require source-level resolution: {full_key}")
    result = frame.pivot(index=KEY, columns=dimensions, values=values)
    # JSON arrays preserve delimiters inside labels, avoiding accidental feature-name collisions.
    result.columns = [prefix + json.dumps(list(t) if isinstance(t, tuple) else [t], separators=(",", ":"))
                      for t in result.columns]
    return result.reset_index()


# Capture every available numeric semiLASER measurement, including both ROIs.
mrs_values = [c for c in raw["mrs"].columns if c not in KEY + ["ROI"]
              and pd.api.types.is_numeric_dtype(raw["mrs"][c])]
mrs_extra = wide_features(numeric_values(raw["mrs"], mrs_values), ["ROI"], mrs_values, "SEMILASER_")
dws_values = ["ADC_log_Glu_Gln_lin", "WCONC_corr", "p_shots_kept", "SN", "linewidth"]
dws_features = (wide_features(raw["dws"], ["ROI"], dws_values, "DWS_")
                if "dws" in raw else pd.DataFrame(columns=KEY))
choice_features = (wide_features(raw["choice"], ["domain", "model", "parameter_name"],
                                 ["parameter_value"], "CHOICE_")
                   if "choice" in raw else pd.DataFrame(columns=KEY))

# Duplicate non-Fatigue ratings are logged and excluded per question, not arbitrarily averaged.
other = raw["ratings"].loc[~raw["ratings"].question.eq("Fatigue")].copy()
ambiguous_other_ratings = other.loc[other.duplicated(KEY + ["question"], keep=False)].copy()
other = other.loc[~other.duplicated(KEY + ["question"], keep=False)]
rating_features = wide_features(other, ["question"], ["rating"], "RATING_")

if "pupil" in raw:
    pupil = raw["pupil"].copy()
    pupil_columns = sorted([c for c in pupil if c.startswith("pupil_") and c[6:].isdigit()],
                           key=lambda c: int(c[6:]))
    if len(pupil_columns) < 4:
        raise ValueError("At least four numbered pupil samples per trial are required.")
    pupil = numeric_values(pupil, pupil_columns + ["response_time", "choiceSS", "ntask_perf"])
    if not pupil.choiceSS.dropna().isin([0, 1]).all():
        raise ValueError("Pupil choiceSS must be 0/1; check its coding before taking means.")
    pupil.loc[pupil.response_time.le(0), "response_time"] = np.nan
    samples = pupil[pupil_columns]
    quarter = max(1, len(pupil_columns) // 4)
    pupil["pupil_mean"] = samples.mean(axis=1)
    pupil["pupil_sd"] = samples.std(axis=1)
    pupil["pupil_coverage"] = samples.notna().mean(axis=1)
    pupil["pupil_end_minus_start"] = samples.iloc[:, -quarter:].mean(axis=1) - samples.iloc[:, :quarter].mean(axis=1)
    # Aggregate within domain to preserve domain-specific choice and pupil differences.
    pupil_session = pupil.groupby(KEY + ["domain"], dropna=False).agg(
        pupil_mean=("pupil_mean", "mean"), pupil_sd=("pupil_sd", "mean"),
        pupil_coverage=("pupil_coverage", "mean"), pupil_change=("pupil_end_minus_start", "mean"),
        median_rt=("response_time", "median"), choice_fraction=("choiceSS", "mean"),
        ntask_performance=("ntask_perf", "mean"), n_trials=("subject_id", "size")).reset_index()
    pupil_values = [c for c in pupil_session if c not in KEY + ["domain"]]
    pupil_features = wide_features(pupil_session, ["domain"], pupil_values, "PUPIL_")

else:
    pupil_features = pd.DataFrame(columns=KEY)

modality_tables = {"ratings_extra": rating_features, "semilaser_extra": mrs_extra,
                   "dws": dws_features, "pupil": pupil_features, "choice": choice_features}
FEATURE_SOURCE = {c: name for name, table in modality_tables.items() for c in table if c not in KEY}
modality_tables = {name: table for name, table in modality_tables.items() if len(table.columns) > len(KEY)}
for name, table in modality_tables.items():
    # Left joins preserve every valid transition, including rows with absent modalities.
    sessions = sessions.merge(table, on=KEY, how="left", validate="one_to_one")
    model_data = model_data.merge(table, on=KEY, how="left", validate="one_to_one")
EXTRA_FEATURES = list(FEATURE_SOURCE)
CHOICE_FEATURES = [c for c, source in FEATURE_SOURCE.items() if source == "choice"]
coverage_rows = []
for name, table in modality_tables.items():
    columns = [c for c in table if c not in KEY]
    coverage_rows.append({"modality": name, "source_session_rows": len(table), "engineered_features": len(columns),
                          "transitions_with_any_measurement": int(model_data[columns].notna().any(axis=1).sum()),
                          "mean_feature_availability": float(model_data[columns].notna().mean().mean())})
modality_coverage = pd.DataFrame(coverage_rows)
display(modality_coverage)
source_column_inventory = pd.DataFrame([
    {"source": name, "column": column, "dtype": str(frame[column].dtype),
     "nonmissing": int(frame[column].notna().sum()), "rows": len(frame)}
    for name, frame in raw.items() for column in frame.columns])
display(source_column_inventory)

## 3. Descripción de features y patrones de datos
Las trayectorias de matplotlib, las distribuciones/correlaciones de seaborn y el Q–Q de SciPy son descriptivos. Se exporta la matriz completa y se muestra una selección legible. Las observaciones repetidas no son independientes; las correlaciones no sirven como prueba confirmatoria ni para elegir el predictor principal después de ver p-valores.

Se conserva Glu/tCr PFC por la pregunta de investigación, sin selección automática de significación. Se necesita distinguir un cociente de metabolitos de una concentración absoluta.

In [ ]:
BASE = ["fatigue", "session", "condition"]
BEHAVIOR = ["accuracy", "median_rt", "within_session_rt_change", "within_session_accuracy_change"]
CANDIDATES = BEHAVIOR + ["mean_rt", "rt_std", "switch_fraction", "early_accuracy", "late_accuracy", "early_rt", "late_rt"]
GLU = "MRS_Glu_tCr_PFC"
FEATURES = {"M0": BASE, "M1": BASE + BEHAVIOR, "M2": BASE + [GLU], "M3": BASE + BEHAVIOR + [GLU]}
PRIMARY_FEATURES = list(dict.fromkeys(sum(FEATURES.values(), [])))
multimodal = CANDIDATES + ["n_trials", GLU] + EXTRA_FEATURES
if not INCLUDE_CHOICE_PARAMETERS:
    multimodal = [c for c in multimodal if c not in CHOICE_FEATURES]
FEATURES["M4_all_modalities"] = list(dict.fromkeys(BASE + multimodal))
AUDIT_FEATURES = list(dict.fromkeys(BEHAVIOR + [GLU] + EXTRA_FEATURES))
AUDIT_SOURCES = {c: FEATURE_SOURCE.get(c, "semilaser_extra" if c == GLU else "behavior") for c in AUDIT_FEATURES}
DESCRIPTIVE = list(dict.fromkeys(["fatigue"] + CANDIDATES + [GLU] + EXTRA_FEATURES))
# Outlier burden is balanced across modalities so hundreds of pupil samples or parameters
# cannot outweigh a smaller modality simply because it supplies more columns.
feature_summary = sessions[DESCRIPTIVE].describe().T
feature_summary["missing_fraction"] = sessions[DESCRIPTIVE].isna().mean()
display(feature_summary.round(3))
# A representative distribution from each added source makes all modalities visible.
fig, axes = plt.subplots(1, len(modality_tables), figsize=(4 * len(modality_tables), 3))
for ax, source in zip(axes, modality_tables):
    choices = [c for c, name in FEATURE_SOURCE.items() if name == source]
    if choices:
        feature = choices[0]
        sns.histplot(sessions[feature].dropna(), ax=ax)
        ax.set_title(source)
        ax.set_xlabel(feature, fontsize=7)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=modality_coverage, x="modality", y="mean_feature_availability", ax=ax)
ax.set(ylim=(0, 1), ylabel="Mean feature availability across transitions",
       title="Structural missingness by source")
plt.tight_layout(); plt.show()
# matplotlib: individual trajectories expose abrupt changes hidden by a group mean.
fig, ax = plt.subplots(figsize=(10, 4))
for _, part in fatigue.groupby(["subject_id", "condition"]):
    ax.plot(part["session"], part["fatigue"], alpha=0.25, marker=".")
ax.set(xlabel="Session", ylabel="Fatigue (source scale)", title="Individual fatigue trajectories")
plt.show()
# seaborn: rank correlations tolerate non-Gaussian feature distributions.
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(sessions[["fatigue"] + BEHAVIOR + [GLU] + EXTRA_FEATURES[:12]].corr(method="spearman"), center=0, vmin=-1, vmax=1,
            cmap="vlag", ax=ax)
ax.set_title("Descriptive correlations (readable subset; full matrix exported)")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=model_data, x="delta_fatigue", hue="condition", element="step", ax=axes[0])
sns.scatterplot(data=model_data, x=GLU, y="delta_fatigue", hue="condition", ax=axes[1])
plt.tight_layout(); plt.show()
# Third graphic: scipy's normal probability plot diagnoses heavy tails, not independence or causality.
fig, ax = plt.subplots(figsize=(6, 4))
if len(model_data) >= 3:
    stats.probplot(model_data["delta_fatigue"], dist="norm", plot=ax)
ax.set_title("SciPy Q–Q plot: fatigue transitions")
plt.show()

## 4. Auditoría de outliers para análisis de sensibilidad
Se conserva la regla de limpieza utilizada por el usuario en M0–M3: cuatro features conductuales y Glu/tCr PFC. Los límites robustos usan mediana/MAD; con MAD cero se usa IQR y, sin dispersión estimable, no se fuerza una clasificación. Se excluyen sujetos con más del 20% de celdas observadas marcadas y al menos ocho observadas. Después se calculan una sola vez los cambios de fatiga extremos (|robust-z|>4.5) entre los sujetos restantes y se eliminan **todos** los candidatos, sin límite de uno. El recuento de transiciones corresponde a esa etapa; las filas ya excluidas por sujeto se registran en la auditoría de sujetos.

Las fuentes adicionales se describen y estudian en la sección 8; no se cambia la regla de limpieza al añadirlas. Esta es una definición operacional de sensibilidad, no prueba de errores de medición ni covariables adicionales del modelo principal. Las exclusiones se aplican a la sensibilidad, mientras el análisis principal conserva todas las transiciones temporalmente válidas con variables requeridas observadas.


In [ ]:
# Conservar la regla de limpieza ya usada en M0–M3 para comparar sensibilidades.
# Las demás modalidades siguen disponibles para descripción y asociaciones complementarias.
AUDIT_FEATURES = BEHAVIOR + [GLU]

def robust_location_scale(frame):
    center = frame.median()
    scale = 1.4826 * frame.sub(center).abs().median()
    fallback = (frame.quantile(.75) - frame.quantile(.25)) / 1.349
    scale = scale.where(scale.gt(0), fallback)
    return center, scale.where(scale.gt(0), np.inf)


def filter_training(train, session_pool, burden=SUBJECT_BURDEN, remove_extreme=True):
    """Aplicar la regla previa: sujetos primero; todos los extremos después, sin límite."""
    pool = session_pool.loc[session_pool.subject_id.isin(train.subject_id.unique())].copy()
    center, scale = robust_location_scale(pool[AUDIT_FEATURES])
    observed = pool[AUDIT_FEATURES].notna()
    flags = pool[AUDIT_FEATURES].sub(center).div(scale).abs().gt(OUTLIER_Z) & observed
    counts = pd.DataFrame({"subject_id": pool.subject_id, "observed": observed.sum(axis=1),
                           "flagged": flags.sum(axis=1)}).groupby("subject_id")[["observed", "flagged"]].sum()
    counts["burden"] = counts.flagged / counts.observed.replace(0, np.nan)
    counts["enough_evidence"] = counts.observed.ge(MIN_OBSERVED_CELLS)
    counts["excluded"] = counts.enough_evidence & counts.burden.gt(burden)
    rejected = counts.index[counts.excluded]
    kept = train.loc[~train.subject_id.isin(rejected)].copy()
    audit = kept[["row_id"] + KEY + ["delta_fatigue"]].copy()
    c, scale_delta = robust_location_scale(kept[["delta_fatigue"]])
    audit["reference_median"] = c.iloc[0]
    audit["reference_scale"] = scale_delta.iloc[0]
    audit["robust_z"] = kept.delta_fatigue.sub(c.iloc[0]).div(scale_delta.iloc[0]).abs()
    audit["candidate"] = audit.robust_z.gt(EXTREME_TRANSITION_Z)
    audit["excluded"] = audit.candidate & bool(remove_extreme)
    # La máscara incluye todos los candidatos; no usar candidates.iloc[0].
    kept = kept.loc[~kept.row_id.isin(audit.loc[audit.excluded, "row_id"])].copy()
    return kept, counts.reset_index(), audit


full_clean, subject_audit, transition_audit = filter_training(model_data, sessions)
display(subject_audit.sort_values("burden", ascending=False))
display(transition_audit.sort_values("robust_z", ascending=False).head(10))
print("Sujetos excluidos en sensibilidad:", int(subject_audit.excluded.sum()))
print("Candidatos tras excluir sujetos:", int(transition_audit.candidate.sum()),
      "; transiciones extremas excluidas:", int(transition_audit.excluded.sum()))
print("El modelo principal NO utiliza estas exclusiones por valor del resultado.")


## 5. Modelo longitudinal principal
Se plantea un intercepto aleatorio por sujeto:

`fatiga futura ~ fatiga actual + sesión + condición + Glu/tCr PFC + (1 | sujeto)`

Se ajusta por REML y se documentan convergencia, varianza aleatoria y advertencias. Una convergencia nominal no basta: varianza próxima a cero, Hessiano no definido positivo o covarianza inválida impiden interpretar sus intervalos como un ajuste mixto fiable. Si ocurre, se informa la limitación y se usa como alternativa explícita una regresión con errores CR1 agrupados por sujeto, corrección de muestra y referencia t con G−1 grados de libertad. CR1 tampoco garantiza inferencia exacta con pocos sujetos.

Los coeficientes se expresan en puntos de fatiga futura por +10 puntos de fatiga actual o +0.1 unidades Glu/tCr. La sesión entra como tendencia lineal y la condición como categoría. No se incorporan interacciones ni una batería de predictores a este modelo pequeño. Se usan casos completos para sus cinco variables; las otras modalidades no determinan la inclusión. La falta de datos puede sesgar la muestra.

**Estimando:** asociación condicional combinada intra/interindividual; no efecto causal ni efecto puramente dentro de un sujeto. Los intervalos del mixto, si es admisible, son aproximaciones asintóticas; las pruebas t de la regresión agrupada y el bootstrap se muestran para juzgar estabilidad. No se interpreta el R² ajustado a la muestra como precisión OOF.

In [ ]:
PRIMARY_COLUMNS = ["subject_id", "condition", "session", "fatigue", "fatigue_future", GLU]
primary = model_data.dropna(subset=PRIMARY_COLUMNS).copy()
if primary.subject_id.nunique() < 10:
    raise ValueError("Muy pocos sujetos para esta inferencia agrupada; revisar el diseño.")
CENTERS = {"fatigue": float(primary.fatigue.mean()), "session": float(primary.session.mean()),
           "glu": float(primary[GLU].mean())}


def inference_frame(frame):
    """Unidades interpretables, con centros fijos para comparar las sensibilidades."""
    out = frame.copy()
    out["fatigue_10"] = (out.fatigue - CENTERS["fatigue"]) / 10
    out["session_c"] = out.session - CENTERS["session"]
    out["glu_01"] = (out[GLU] - CENTERS["glu"]) / .1
    return out


primary = inference_frame(primary)
FORMULA = "fatigue_future ~ fatigue_10 + session_c + C(condition) + glu_01"
BASE_FORMULA = "fatigue_future ~ fatigue_10 + session_c + C(condition)"
counts_per_subject = primary.groupby("subject_id").size()
attrition = pd.DataFrame([
    {"sample": "transiciones válidas", "n": len(model_data), "subjects": model_data.subject_id.nunique()},
    {"sample": "modelo principal", "n": len(primary), "subjects": primary.subject_id.nunique()},
])
display(attrition)
print("Sujetos con más de una transición:", int(counts_per_subject.gt(1).sum()))


def clustered_fit(frame, formula=FORMULA):
    return smf.ols(formula, data=frame).fit(
        cov_type="cluster", cov_kwds={"groups": frame.subject_id,
                                     "use_correction": True, "df_correction": True}, use_t=True)


def coefficient_table(fit, method, mixed=False):
    terms = list(fit.fe_params.index) if mixed else list(fit.params.index)
    ci = fit.conf_int().loc[terms]
    return pd.DataFrame({"term": terms, "estimate": fit.params.loc[terms].to_numpy(),
                         "ci_low": ci.iloc[:, 0].to_numpy(), "ci_high": ci.iloc[:, 1].to_numpy(),
                         "p_value": fit.pvalues.loc[terms].to_numpy(), "method": method})


mixed_attempts, mixed_result = [], None
for optimizer in ["lbfgs", "powell"]:
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            candidate = smf.mixedlm(FORMULA, primary, groups=primary.subject_id).fit(
                reml=True, method=optimizer, maxiter=2000, disp=False)
        messages = [str(item.message) for item in caught]
        ratio = float(candidate.cov_re.iloc[0, 0] / candidate.scale)
        cov = candidate.cov_params().loc[candidate.fe_params.index, candidate.fe_params.index]
        covariance_ok = bool(np.isfinite(cov.to_numpy()).all() and
                             np.linalg.eigvalsh(cov.to_numpy()).min() > 0)
        problematic = any("not positive definite" in message.lower() or "singular" in message.lower()
                          for message in messages)
        admissible = bool(candidate.converged and ratio > 1e-6 and covariance_ok and not problematic)
        mixed_attempts.append({"optimizer": optimizer, "converged": bool(candidate.converged),
                               "random_intercept_variance": float(candidate.cov_re.iloc[0, 0]),
                               "random_residual_ratio": ratio, "admissible": admissible,
                               "warnings": messages})
        if admissible:
            mixed_result = candidate
            break
    except (ValueError, np.linalg.LinAlgError) as error:
        mixed_attempts.append({"optimizer": optimizer, "admissible": False,
                               "error": str(error)})
clustered_result = clustered_fit(primary)
clustered_coefficients = coefficient_table(clustered_result, "OLS CR1 + t(G−1)")
if mixed_result is None:
    principal_method = "Regresión OLS con errores CR1 agrupados por sujeto y t(G−1)"
    principal_coefficients = clustered_coefficients.copy()
    print("El intercepto aleatorio no se estima de forma fiable; se usa la alternativa agrupada.")
else:
    principal_method = "Modelo mixto REML con intercepto aleatorio por sujeto"
    principal_coefficients = coefficient_table(mixed_result, "MixedLM REML, IC asintótico", mixed=True)
print("Método principal:", principal_method)
print("Grados de libertad de referencia agrupada:", primary.subject_id.nunique() - 1)
display(pd.DataFrame(mixed_attempts))
display(principal_coefficients.round(4))
if mixed_result is not None:
    display(clustered_coefficients.round(4))

# Dibujar coeficientes e incertidumbre; nunca presentarlos como scores predictivos.
plot_table = principal_coefficients.loc[principal_coefficients.term.ne("Intercept")]
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(plot_table.estimate, np.arange(len(plot_table)),
            xerr=np.vstack([plot_table.estimate - plot_table.ci_low,
                            plot_table.ci_high - plot_table.estimate]), fmt="o", capsize=4)
ax.axvline(0, color="black", linestyle="--")
ax.set_yticks(np.arange(len(plot_table)), plot_table.term)
ax.set(xlabel="Coeficiente e IC 95% (puntos de fatiga futura)", title="Asociaciones ajustadas")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(clustered_result.fittedvalues, clustered_result.resid)
axes[0].axhline(0, linestyle="--", color="black")
axes[0].set(xlabel="Ajuste en la muestra", ylabel="Residuo", title="Diagnóstico, no predicción OOF")
stats.probplot(clustered_result.resid, plot=axes[1])
plt.tight_layout(); plt.show()

## 6. t de Student pareada: cambio entre sesiones fijas
Se compara Fatigue en sesión 0 y sesión 5 usando exclusivamente sujetos con ambas mediciones inequívocas. Las sesiones se fijan en configuración antes del contraste: no se elige para cada sujeto el par que más cambia. Cada persona aporta una sola diferencia y se exige la misma condición en ambos extremos.

La t pareada evalúa si la **media de las diferencias** es cero. Se informa n de sujetos, cambio medio, IC t 95%, t, grados de libertad, p bilateral y d_z. La normalidad relevante es la de las diferencias entre sujetos, no la de cada sesión. El contraste no prueba que el glutamato prediga fatiga. Wilcoxon se añade como comprobación complementaria con hipótesis distinta (simetría/localización), no como sustitución automática elegida por p-valores de normalidad.

In [ ]:
paired_wide = fatigue.pivot(index=["subject_id", "condition"], columns="session", values="fatigue")
if PAIR_START not in paired_wide or PAIR_END not in paired_wide:
    raise ValueError("Las sesiones fijadas para la comparación pareada no existen.")
pairs = paired_wide[[PAIR_START, PAIR_END]].dropna().reset_index()
if pairs.subject_id.duplicated().any():
    raise ValueError("Hay más de un par por persona: definir un contraste de condición antes de la t.")
if len(pairs) < 3:
    raise ValueError("Se necesitan al menos tres pares para este análisis.")
differences = pairs[PAIR_END] - pairs[PAIR_START]
paired_test = stats.ttest_rel(pairs[PAIR_END], pairs[PAIR_START])
paired_ci = paired_test.confidence_interval(confidence_level=.95)
sd_difference = differences.std(ddof=1)
paired_summary = pd.DataFrame([{
    "start_session": PAIR_START, "end_session": PAIR_END, "n_subjects": len(pairs),
    "mean_change": differences.mean(), "ci_low": paired_ci.low, "ci_high": paired_ci.high,
    "t": paired_test.statistic, "df": len(pairs)-1, "p_two_sided": paired_test.pvalue,
    "paired_dz": differences.mean()/sd_difference if sd_difference > 0 else np.nan,
}])
display(paired_summary.round(4))
if differences.ne(0).any():
    wilcoxon = stats.wilcoxon(differences, alternative="two-sided", method="auto")
    print("Wilcoxon complementario:", wilcoxon)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(differences, ax=axes[0])
axes[0].set(xlabel="Fatiga final − inicial", title="Una diferencia por sujeto")
stats.probplot(differences, plot=axes[1])
plt.tight_layout(); plt.show()

## 7. Sensibilidad y bootstrap por sujeto
Se repite la regresión agrupada en cuatro muestras: sin exclusiones, solo sujetos, solo extremos y ambas reglas. Se muestran n, sujetos y asociación de glutamato. El tamaño muestral y la población seleccionada cambian; un coeficiente o p diferente no demuestra que limpiar haya mejorado la inferencia.

El bootstrap remuestrea **sujetos completos**, conservando juntas sus transiciones. Se reajusta OLS y se obtiene un IC percentil del coeficiente, no se remuestrean trials como independientes. Los casos completos y la regla de inclusión principal se mantienen fijos en este bootstrap; no incorpora incertidumbre por datos ausentes, selección del protocolo ni selección de outliers. Se registran ajustes fallidos y no se esconden.

La descomposición intra/interindividual es exploratoria: usa medias de glutamato calculadas en las observaciones disponibles de cada sujeto. Puede ayudar a distinguir asociaciones, pero la parte intraindividual depende de pocos sujetos repetidos y esas medias no son features prospectivas de una predicción t→t+1.

In [ ]:
sensitivity_rows = []
for label, burden, extremes in [("sin_exclusiones", 1.0, False),
                                 ("solo_sujetos", SUBJECT_BURDEN, False),
                                 ("solo_extremos", 1.0, True),
                                 ("ambas_reglas", SUBJECT_BURDEN, True)]:
    retained, _, _ = filter_training(model_data, sessions, burden, extremes)
    part = inference_frame(retained.dropna(subset=PRIMARY_COLUMNS))
    fit = clustered_fit(part)
    coefficient = coefficient_table(fit, "CR1+t").set_index("term").loc["glu_01"]
    sensitivity_rows.append({"scenario": label, "n": len(part), "subjects": part.subject_id.nunique(),
                             "glu_estimate": coefficient.estimate, "ci_low": coefficient.ci_low,
                             "ci_high": coefficient.ci_high, "p_value": coefficient.p_value})
statistical_sensitivity = pd.DataFrame(sensitivity_rows)
display(statistical_sensitivity.round(4))

rng = np.random.default_rng(SEED)
subject_blocks = {sid: part for sid, part in primary.groupby("subject_id")}
subject_ids = list(subject_blocks)
bootstrap_coefficients, bootstrap_failures = [], []
for repeat in range(BOOTSTRAP_REPEATS):
    sampled = rng.choice(subject_ids, len(subject_ids), replace=True)
    sample = pd.concat([subject_blocks[sid] for sid in sampled], ignore_index=True)
    design = smf.ols(FORMULA, data=sample)
    if np.linalg.matrix_rank(design.exog) < design.exog.shape[1]:
        bootstrap_failures.append({"repeat": repeat, "reason": "rank_deficient"})
        continue
    fit = design.fit()
    estimate = float(fit.params["glu_01"])
    if not np.isfinite(estimate):
        bootstrap_failures.append({"repeat": repeat, "reason": "nonfinite"})
        continue
    bootstrap_coefficients.append({"repeat": repeat, "glu_01": estimate})
bootstrap_coefficients = pd.DataFrame(bootstrap_coefficients)
if len(bootstrap_coefficients) < .9 * BOOTSTRAP_REPEATS:
    warnings.warn("Más del 10% de remuestreos falló; no interpretar el IC bootstrap sin revisar.")
bootstrap_ci = bootstrap_coefficients.glu_01.quantile([.025, .975])
bootstrap_summary = pd.DataFrame([{"parameter": "glu_01", "repeats_requested": BOOTSTRAP_REPEATS,
                                  "successful": len(bootstrap_coefficients),
                                  "failed": len(bootstrap_failures),
                                  "percentile_ci_low": bootstrap_ci.loc[.025],
                                  "percentile_ci_high": bootstrap_ci.loc[.975]}])
display(bootstrap_summary)

within_between = primary.copy()
within_between["glu_subject_mean"] = within_between.groupby("subject_id")[GLU].transform("mean")
within_between["glu_within_01"] = (within_between[GLU] - within_between.glu_subject_mean)/.1
within_between["glu_between_01"] = (within_between.glu_subject_mean - CENTERS["glu"])/.1
wb_formula = "fatigue_future ~ fatigue_10 + session_c + C(condition) + glu_within_01 + glu_between_01"
wb_design = smf.ols(wb_formula, data=within_between)
if np.linalg.matrix_rank(wb_design.exog) == wb_design.exog.shape[1]:
    within_between_result = clustered_fit(within_between, wb_formula)
    within_between_coefficients = coefficient_table(within_between_result, "CR1+t; exploratorio")
else:
    within_between_coefficients = pd.DataFrame()
    print("Descomposición no identificable con esta muestra.")
display(within_between_coefficients)

## 8. Otras modalidades: asociaciones exploratorias
Cada feature se estudia por separado, ajustando fatiga actual, sesión y condición. No se utiliza un algoritmo de selección para elegir el modelo principal. Se exige un mínimo de 20 filas y 15 sujetos, variación y matriz de diseño de rango completo; cada omisión se registra. Se informa efecto por una desviación estándar observada de esa feature y el tamaño de su muestra.

Se aplica Holm a todos los contrastes complementarios evaluables de esta familia. Los IC mostrados son nominales, no simultáneos; los p corregidos no eliminan sesgos por datos ausentes, decisiones previas o parámetros estimados upstream. Las muestras entre modalidades pueden diferir, por lo que estos coeficientes no son una clasificación de qué modalidad es mejor. Métricas de calidad/movimiento describen calidad, no mecanismos de fatiga. Las columnas del archivo de pupila aparecen aquí solo cuando se pudo cargar ese archivo.

In [ ]:
screen_features = list(dict.fromkeys(CANDIDATES + ["n_trials"] + EXTRA_FEATURES))
screen_rows, screen_skipped = [], []
for feature in screen_features:
    part = model_data.dropna(subset=["fatigue_future", "fatigue", "session", "condition", feature]).copy()
    reason = None
    if len(part) < MIN_SCREEN_ROWS or part.subject_id.nunique() < MIN_SCREEN_SUBJECTS:
        reason = "insufficient_rows_or_subjects"
    elif part[feature].nunique() <= 1:
        reason = "constant"
    # Avoid retesting a duplicate of the primary physiological predictor under another name.
    elif feature in EXTRA_FEATURES and part[GLU].notna().all() and np.allclose(part[feature], part[GLU]):
        reason = "duplicate_primary_Glu"
    if reason:
        screen_skipped.append({"feature": feature, "reason": reason})
        continue
    part["fatigue_10"] = (part.fatigue - CENTERS["fatigue"])/10
    part["session_c"] = part.session - CENTERS["session"]
    part["screen_z"] = (part[feature] - part[feature].mean())/part[feature].std(ddof=1)
    formula = BASE_FORMULA + " + screen_z"
    design = smf.ols(formula, data=part)
    if np.linalg.matrix_rank(design.exog) < design.exog.shape[1]:
        screen_skipped.append({"feature": feature, "reason": "rank_deficient"})
        continue
    fit = clustered_fit(part, formula)
    ci = fit.conf_int().loc["screen_z"]
    screen_rows.append({"feature": feature, "source": FEATURE_SOURCE.get(feature, "n_task"),
                        "n": len(part), "subjects": part.subject_id.nunique(),
                        "effect_per_SD": fit.params["screen_z"], "ci_low_nominal": ci.iloc[0],
                        "ci_high_nominal": ci.iloc[1], "p_raw": fit.pvalues["screen_z"]})
modality_associations = pd.DataFrame(screen_rows)
if len(modality_associations):
    finite = np.isfinite(modality_associations.p_raw)
    modality_associations["p_holm"] = np.nan
    modality_associations.loc[finite, "p_holm"] = multipletests(
        modality_associations.loc[finite, "p_raw"], method="holm")[1]
    display(modality_associations.sort_values(["source", "feature"]).round(4))
else:
    print("No hay contrastes complementarios evaluables.")
display(pd.DataFrame(screen_skipped))

## 9. Exportaciones e interpretación
Un IC estrecho y estable es más informativo que perseguir p<0.05. Un IC que incluya cero no demuestra ausencia de asociación; indica qué valores siguen siendo compatibles con el análisis. Un IC que excluya cero tampoco demuestra causalidad ni predicción fiable, especialmente tras los análisis exploratorios anteriores.

La t pareada describe cambio medio entre dos sesiones. La regresión estima asociaciones ajustadas. La evaluación OOF del notebook v2 estima errores al predecir sujetos no vistos. Son preguntas diferentes y sus resultados no se sustituyen entre sí.

Revisar: disponibilidad por fuente, número de sujetos repetidos, diagnóstico mixto, unidad del efecto, IC agrupado/bootstrap, estabilidad tras exclusiones y separación intra/interindividual. No afirmar que el marcador anticipa fatiga antes de ser conscientemente reportada sin confirmar el orden de adquisición y el diseño.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
exports = {
    "attrition": attrition, "principal_coefficients": principal_coefficients,
    "clustered_coefficients": clustered_coefficients, "paired_t_test": paired_summary,
    "statistical_sensitivity": statistical_sensitivity, "bootstrap_summary": bootstrap_summary,
    "bootstrap_coefficients": bootstrap_coefficients,
    "within_between_coefficients": within_between_coefficients,
    "modality_associations": modality_associations, "modality_coverage": modality_coverage,
    "source_column_inventory": source_column_inventory,
    "feature_summary": feature_summary.reset_index(names="feature"),
    "subject_exclusion_audit": subject_audit, "transition_exclusion_audit": transition_audit,
    "ambiguous_fatigue_ratings": ambiguous, "ambiguous_other_ratings": ambiguous_other_ratings,
    "skipped_modality_tests": pd.DataFrame(screen_skipped),
    "bootstrap_failures": pd.DataFrame(bootstrap_failures),
}
for name, table in exports.items():
    table.to_csv(RESULTS_DIR / f"{name}.csv", index=False)
manifest = {"analysis": "exploratory longitudinal association", "principal_method": principal_method,
            "formula": FORMULA, "centers": CENTERS, "versions": VERSIONS,
            "inputs": provenance, "missing_optional_sources": missing_optional,
            "mixed_attempts": mixed_attempts, "primary_n": len(primary),
            "primary_subjects": primary.subject_id.nunique(), "repeated_subjects": int(counts_per_subject.gt(1).sum()),
            "primary_outcome_exclusion": False,
            "sensitivity_rules": {"feature_z": OUTLIER_Z, "subject_burden": SUBJECT_BURDEN,
                                  "min_cells": MIN_OBSERVED_CELLS, "transition_z": EXTREME_TRANSITION_Z,
                                  "transition_policy": "all_above_threshold_after_subject_exclusion",
                                  "feature_pool": AUDIT_FEATURES, "burden_weighting": "observed_cell_fraction"},
            "bootstrap_repeats": BOOTSTRAP_REPEATS, "seed": SEED,
            "paired_sessions": [PAIR_START, PAIR_END], "multiple_testing": "Holm: complementary modality tests"}
(RESULTS_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Resultados estadísticos guardados en", RESULTS_DIR.resolve())
print("No se presentan estas asociaciones como mejoras en RMSE ni como predicción OOF.")